# Reports as artifacts

Reports can be stored with a dataset, shared in an issue, passed to an
agent, or compared across runs. This chapter covers JSON, Markdown, the
context-budgeted digest, the Python dict, and the conditions under which
serialization is deterministic.
The field-level reference is [](../appendix/report_contract.ipynb).

The fixture contains a region-code mapping and a column equal to twice the quantity:

In [1]:
import json
import random

import polars as pl

import onceover

rng = random.Random(4)
n = 800

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["red", "green", "blue"][r] for r in region_id]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": list(range(n)),
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})

report = onceover.profile(df)
print(report.overview["headline"])

800 rows × 5 columns (standard budget); key candidate: order_id; 0 functional dependencies, 4 predictive dependencies, 6 conditional rules


## JSON

`to_json()` returns the complete report — overview, columns, findings,
groups, selection summary, coverage — as one deterministic, schema-versioned
string. `report.data` parses it into a dict; `save(path)` writes it;
`json.dumps(report.data, indent=2)` formats it for reading.
`schema_version` (currently `"0.8"`) identifies the report format. Changes are usually additive: the 0.7 → 0.8 revision renamed parts of the vocabulary — detectors, finding types, the column-card field, one facet field — without changing what any statement means; the 0.6 → 0.7 revision added an `associations`
table holding every pair the association screens tested, with its
effect sizes, raw and corrected p-values, and reporting tier; the
0.5 → 0.6 revision moved restatements of one exact relation into
`grouped_findings` under a lead finding; the 0.4 → 0.5 revision added
the frame account — the frame's `grain` and `dominant` structure — to
the `frame` section; the 0.3 → 0.4 revision added a `frame` section
and `semantic_role` fields on the column cards and left findings
unchanged; the 0.2 → 0.3 revision added a `facets` object to
every finding and left the category and type values unchanged. The
0.1 → 0.2 revision, by contrast, changed the finding taxonomy
incompatibly. Readers should check the schema version and tolerate
unknown fields and categories.

## Determinism

With the same package version, input, and settings, repeated runs produce
identical JSON when no deadline is set and timing is disabled. A deadline can change which passes
finish. Coverage records those skips.

Two different tables can produce the same summary, so matching report JSON
does not prove that the input data is identical.

In [2]:
first = onceover.profile(df)
second = onceover.profile(df, budget="standard")

print("byte-identical:", first.to_json() == second.to_json())
print("schema_version:", report.data["schema_version"])
print("selection:", report.data["selection"])

byte-identical: True
schema_version: 0.8
selection: {'candidates': 50, 'verified_candidates': 40, 'distinct_statements': 39, 'selected': 35, 'families': 2, 'grouped_statements': 4}


`selection.candidates` counts the detectors' candidates before consolidation.
`selection.distinct_statements` counts the different claims after matching
candidates are combined; `selection.selected` counts the findings retained
under the selection limits. Compare these counts when a report contains
fewer findings than the detectors produced.

## Markdown

`to_markdown()` renders the blocks as a compact document — headline, ranked
findings with their basis and form, the related-finding groups, a column
table, and the coverage table. It is the format meant for pasting into a
pull request or handing to a person:

In [3]:
print(report.to_markdown())

# onceover report

800 rows × 5 columns (standard budget); key candidate: order_id; 0 functional dependencies, 4 predictive dependencies, 6 conditional rules

## Overview

- **rows × columns**: 800 × 5
- **dtypes**: 4×int64, 1×string
- **roles**: 4×categorical, 1×unique

## Frame

- **grain**: order_id uniquely identifies rows
- **semantic roles**: order_id: row_number, region_id: dimension, region_name: dimension, quantity: dimension, double_quantity: dimension

## Findings

- **[0.95·measured] uniqueness/uniqueness** — order_id uniquely identifies rows (primary key candidate)
  - evidence: keys (uniqueness), direct_comparison, 800 rows evaluated, full data
    - metrics: distinct=800, n=800, plausibility_score=1
- **[0.8695·measured] dependencies/affine (affine)** — double_quantity = 2.000000·quantity within tolerance on all 800 non-missing row pairs
  - evidence: identities (ols_residual_scan), direct_comparison, 800 rows evaluated, full data
    - metrics: a=2, absolute_tolerance=1

## Digest

`report.to_digest(budget_chars=4_000)` renders a plain-text briefing sized to a
reading budget — built for the case where the report's JSON is larger
than the context it has to fit into. The briefing is organised by the
questions a reader brings to a new table, in sections, each with its
own share of the budget. *What a row is*: the grain statement (the
frame's key, factorized — `12 stations × 35,064 timestamps` — when
the frame crosses its dimensions with its time axis, or the near key
with its duplicate count when the axis nearly identifies rows) and
the time axis with its measured span and step. *What is wrong*:
missingness and its structure, duplicates, sentinels, impossible
values, and formats, opened by a line of exact completeness counts.
*What is redundant*: one line per family of findings that restate
one another. *What relates to the target*, when a target is
designated: leak columns first, then the strongest drivers.
*Other relationships*, *Anomalies*, and *Not examined*: the passes
that run by default and were capped, cut by the deadline, or
errored — passes that run on request are never listed. A section
that cannot fill its share passes the remainder to the sections
still holding facts, in section order. The what-is-wrong section
keeps its share when a target is designated too: under cap
pressure, selection admits the hygiene reserve's statements ahead
of structure and shape, so designating a target no longer makes
the hygiene findings vanish from the report.

Inside the body, findings are merged into facts before rendering:
one line per distinct set of stated columns. A model-based finding
states its target and the drivers it credits, not every input it
screened. When an exact form of a fact exists — a functional
dependency, an arithmetic identity, an encoding — the exact form
is the line, and the statistical restatements fold into it. Each
line is the representative finding's claim in plain words, its
numbers rounded to three significant figures, followed by that one
finding's id. Inside a relationship section, leak lines lead,
then the predictability directory — one line per column, naming
its whole driver set — when the whole directory fits the section's
share; the remaining lines run breadth-first across kinds, with
break and regime statements along an ordered axis leading each
round, so one detector's repeated pattern does not fill the
section. The findings array in the JSON keeps its own order
(account, completeness, relations, column structure, deviations);
the briefing's section order deliberately differs, because the
briefing is read first, while the array's order is the one stored
reports are compared in. When the lines exceed the budget, each
section keeps what fits its share, strongest first, and the
briefing ends by stating how many facts were omitted and how to
retrieve them; the result never exceeds the budget.

Weak association findings take no line of their own: pairs that
stay significant after correction but fall below the effect bar
collapse into a single line with their count and strongest
examples. Their full evidence remains in the JSON.

A briefing is an index into the report, not a summary of record.
A line states one fact and names the finding that speaks for it;
the findings merged into that fact, and every finding's evidence,
remain in the JSON, reachable through the report.

The digest is also the notebook display. When a cell's result is
the report object, the notebook renders this briefing plus a one-line
footer naming `to_markdown()`, `to_digest(n)` and `select(...)`;
it does not render the full markdown document.


In [4]:
print(report.to_digest(500))

profile digest: 800 rows x 5 columns, 35 findings, schema 0.8
What a row is:
order_id uniquely identifies rows
What is redundant:
double_quantity = 2·quantity within tolerance on all 800 non-missing row pairs [dependencies-b097b740f5504a9d]
... [5 fact(s) omitted to fit 500 chars; 35 findings total: column_roles 4, uniqueness 1; report.select(category="column_roles") retrieves them]



## Feeding an agent

The function below copies each finding's `claim` verbatim and includes its
id and category. It summarizes the evidence record with the largest
`rows_evaluated`, giving its basis, validation method, and row count.
The complete evidence remains available in the report.

In [5]:
def brief(f):
    primary = max(f["evidence"], key=lambda e: e["scope"]["rows_evaluated"] or 0)
    return {
        "id": f["id"],
        "category": f["category"],
        "type": f.get("type"),
        "claim": f["claim"],
        "basis": primary["basis"],
        "validation": primary["validation"]["method"],
        "rows_evaluated": primary["scope"]["rows_evaluated"],
        "interpretation": f.get("interpretation"),
    }

digest = {
    "headline": report.overview["headline"],
    "top_findings": [brief(f) for f in report.findings[:4]],
    "coverage": report.coverage,
}
print(json.dumps(digest["top_findings"][0], indent=2))

{
  "id": "uniqueness-1ac6146d2c27c2e2",
  "category": "uniqueness",
  "type": "uniqueness",
  "claim": "order_id uniquely identifies rows (primary key candidate)",
  "basis": "measured",
  "validation": "direct_comparison",
  "rows_evaluated": 800,
  "interpretation": "use as join key; exclude from numeric features"
}


Keep the basis, row count, and validation method with the claim when
sharing this digest. The record with the most evaluated rows is not
necessarily the best validated: inspect the full evidence before choosing
a result for an automated action. When no finding is reported, check coverage to see whether the detector ran and
what rows and candidates it examined.

## The Rust API

The same profiler is a Rust crate over Polars 0.55, for pipelines that
never leave Rust:

```toml
[dependencies]
onceover = "0.1"
polars = "0.55"
```

```rust
use onceover::{profile, Budget, ProfileConfig};
use polars::prelude::*;

fn main() -> PolarsResult<()> {
    let df = df!("id" => [1i64, 2, 3], "value" => [4.0, 5.0, 6.0])?;

    let cfg = ProfileConfig {
        budget: Budget::Deep,
        ..ProfileConfig::default()
    };
    let report = profile(&df, &cfg);

    println!("{}", report.overview.headline);
    println!("{}", report.to_json());   // same schema, same determinism
    Ok(())
}
```

`ProfileConfig::default()` uses the `standard` budget, default detectors,
the default selection policy, and no deadline; the `selection` field is the
same `SelectionPolicy` that caps findings and rules in Python via
`max_findings`/`max_rules`. The report schema is identical across
languages, so a Rust service and a Python notebook produce comparable
artifacts from the same table.

## Compare profiles

`before.diff(after)` compares statements by stable ID and records added,
removed, and changed claims or scores, plus coverage-status changes. It is
useful after a new extract arrives or a transformation changes the table.
Evidence-only changes with unchanged claim and score are not counted as
changes. This comparison is distinct from the model-routing drift pass.

The diff also compares the column cards, under `columns`. Columns are
matched by name: a column only in the newer profile is added, one only in
the older is removed, and one whose card dtype changed is retyped —
stated distinctly from a removal plus an addition. For a column in both
with an unchanged dtype, the diff lists the card quantities that moved,
each with its before and after values, the absolute change, and the
relative change where one is meaningful. The compared quantities are the
card pass's exact counts — rows, nulls and their share, distinct values —
and, for numeric columns, the mean, minimum, and maximum the card
carries; the card carries no sum, so none is reported. Those quantities
are computed on the full table before any budget gating, so two profiles
of the same data taken at different budgets still diff empty at the card
level. A renamed column reads as a removal plus an addition, and
row-level changes are not compared.

The example appends duplicate rows to the orders fixture:


In [6]:
after = onceover.profile(pl.concat([df, df.head(3)]))
comparison = report.diff(after)
print("Added findings:")
for finding in comparison["added"]:
    print(finding["category"], finding["claim"])
print("Removed:", len(comparison["removed"]), "Changed:", len(comparison["changed"]))
print("Coverage status changes:", comparison["coverage_changes"])
columns = comparison["columns"]
print("Columns added:", [c["name"] for c in columns["added"]],
      "removed:", [c["name"] for c in columns["removed"]],
      "retyped:", [c["name"] for c in columns["retyped"]])
for entry in columns["changed"]:
    moved = {k: (v["before"], v["after"]) for k, v in entry["changes"].items()}
    print("Card changed:", entry["name"], moved)
print("Columns unchanged:", columns["unchanged"])


Added findings:
duplicates 3 rows repeat earlier values across the compared columns (3 groups, largest 2)
Removed: 0 Changed: 16
Coverage status changes: {}
Columns added: [] removed: [] retyped: []
Card changed: double_quantity {'n': (800, 803), 'mean': (11.1875, 11.193026151930262)}
Card changed: order_id {'n': (800, 803), 'mean': (399.5, 398.0112079701121)}
Card changed: quantity {'n': (800, 803), 'mean': (5.59375, 5.596513075965131)}
Card changed: region_id {'n': (800, 803), 'mean': (0.97, 0.9676214196762142)}
Card changed: region_name {'n': (800, 803)}
Columns unchanged: 0


## Budgeted JSON

`to_json(max_chars=...)` retains high-ranked findings in full and indexes
others. If needed it reduces column-card and coverage detail. Its `budget`
record states what was omitted; it raises `ValueError` if the minimal
index cannot fit. Use `to_json()` without a size limit to retain the whole
artifact. This is a presentation budget, separate from discovery budgets.

The [API reference](api.ipynb) lists the report methods and parameters.
